# Notebook 09: Data Leakage

**Difficulty:** ⭐⭐⭐⭐⭐ | **Estimated Time:** 3 hours

**Week 4 — Data Fundamentals & Preprocessing Pipelines**

---

## Why does this matter for ML?

Data leakage is the single most dangerous silent failure in machine learning. A leaky model will:

- Score 99%+ accuracy in testing and fall apart the moment it touches real data
- Cost your organization real money before anyone notices
- Be extremely hard to debug because the numbers *look* correct
- Erode trust in the entire ML team when discovered

In fraud detection, a leaky model might approve every fraudulent transaction in production because it secretly relied on a variable that only exists *after* fraud has already been confirmed. The model never learned to detect fraud — it learned to read the answer sheet.

---

## Real-World Analogy: The Answer Sheet Problem

Imagine a teacher is preparing students for a standardised exam. During practice sessions, the teacher accidentally hands out the actual exam answers alongside the practice problems. Students score 100% in every practice run. The teacher concludes: "My students are brilliantly prepared!"

On exam day — no answer sheet. Students fail spectacularly.

**This is data leakage.** Your model aced the test set because it had information it will never have in the real world. The cruel part: you won't discover this until you've already deployed and real losses are accumulating.

---

## The Four Types of Data Leakage

| Type | Plain English | Fraud Example |
|------|--------------|---------------|
| **Target Leakage** | A feature contains the answer | `is_account_flagged` is set to 1 after fraud is confirmed |
| **Train-Test Contamination** | Test data influences training preprocessing | Fitting a scaler on the full dataset before splitting |
| **Temporal Leakage** | Future information predicts the past | Rolling fraud rate computed using data from the future |
| **Group Leakage** | Same person appears in both train and test | User 12345's transactions split across train and test sets |


In [ ]:
# ============================================================
# CELL 1: Imports and global settings
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Suppress convergence warnings so notebook output stays clean
warnings.filterwarnings('ignore')

# Set a fixed random seed so every run produces the same synthetic data
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Seaborn style for all plots
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('All libraries imported successfully.')
print(f'numpy {np.__version__} | pandas {pd.__version__}')

## Step 1: Build a Synthetic Credit Card Fraud Dataset

We will generate a realistic-looking dataset with:
- 10,000 credit card transactions
- ~5% fraud rate (class imbalance — typical in real fraud data)
- Timestamps so we can demonstrate temporal leakage
- User IDs so we can demonstrate group leakage
- A **deliberately leaky feature** (`resolution_code`) that we will expose and remove

**Key concept:** `resolution_code` is a code that a fraud analyst enters *after* reviewing a flagged transaction. In real operations, this code does not exist until *after* fraud has been confirmed. Including it is like including the exam answer.


In [ ]:
# ============================================================
# CELL 2: Generate synthetic credit card fraud dataset
# ============================================================
n_transactions = 10_000
fraud_rate = 0.05  # 5% of transactions are fraudulent
n_users = 500      # 500 unique card holders

# ---- Generate fraud labels first (the ground truth) ----
is_fraud = np.random.binomial(1, fraud_rate, n_transactions)

# ---- Timestamps: one year of transactions, chronologically sorted ----
# We sort timestamps so we can do time-based splitting later
timestamps = pd.date_range('2023-01-01', periods=n_transactions, freq='1h')

# ---- User IDs: 500 users, each appearing ~20 times on average ----
user_ids = np.random.randint(1, n_users + 1, n_transactions)

# ---- Transaction amount: fraudulent transactions skew higher ----
# Legitimate: mean $85, std $60 | Fraudulent: mean $320, std $200
amount = np.where(
    is_fraud == 1,
    np.random.normal(320, 200, n_transactions).clip(5, 5000),
    np.random.normal(85, 60, n_transactions).clip(1, 2000)
)

# ---- Merchant category: 5 types, fraud more common in 'online' ----
# We encode as integers here; we'll show categorical encoding in Notebook 10
merchant_categories = np.random.choice(
    ['grocery', 'restaurant', 'online', 'gas_station', 'retail'],
    n_transactions,
    p=[0.30, 0.20, 0.25, 0.10, 0.15]  # online is most common for fraud
)

# ---- Behavioural features ----
# hour_of_day: fraudulent transactions peak at night
hour_of_day = np.where(
    is_fraud == 1,
    np.random.choice(np.arange(0, 24), n_transactions,
                     p=[0.08,0.09,0.09,0.08,0.05,0.03,0.02,0.02,0.02,0.02,
                        0.02,0.02,0.03,0.03,0.03,0.04,0.04,0.05,0.05,0.05,
                        0.05,0.05,0.06,0.07]),
    np.random.randint(0, 24, n_transactions)
)

distance_from_home_km = np.where(
    is_fraud == 1,
    np.random.exponential(200, n_transactions).clip(0, 20000),
    np.random.exponential(15, n_transactions).clip(0, 500)
)

# ---- THE LEAKY FEATURE: resolution_code ----
# This code is written by a fraud analyst AFTER they confirm fraud.
# Resolution code 3 = 'confirmed fraud', all others = 'not fraud'.
# At prediction time (when a transaction just occurred) this field is EMPTY.
# Using it is like giving the model the answer.
resolution_code = np.where(
    is_fraud == 1,
    np.random.choice([3, 3, 3, 2], n_transactions),  # code 3 = fraud confirmed
    np.random.choice([0, 1, 4, 1], n_transactions)   # codes 0,1,4 = not fraud
)

# ---- Assemble DataFrame ----
df = pd.DataFrame({
    'transaction_id': np.arange(1, n_transactions + 1),
    'timestamp':      timestamps,
    'user_id':        user_ids,
    'amount':         amount.round(2),
    'merchant_category': merchant_categories,
    'hour_of_day':    hour_of_day,
    'distance_from_home_km': distance_from_home_km.round(1),
    'resolution_code': resolution_code,   # <-- THE LEAKY FEATURE
    'is_fraud':        is_fraud
})

print(f'Dataset shape: {df.shape}')
print(f'Fraud rate:    {df["is_fraud"].mean():.1%}')
print(f'Unique users:  {df["user_id"].nunique()}')
print()
print(df.head(6).to_string())

---
## Type 1: Target Leakage

**Plain English definition:** A feature in your dataset was created *because of* or *at the same time as* the target variable. Including it lets the model cheat — it learns the answer, not the pattern.

**Medical analogy:** Suppose you want to predict which patients have cancer. One of your features is `patient_had_surgery`. The surgery happened *because* the doctor already diagnosed cancer. Using it means you're essentially handing the model the diagnosis before it predicts the diagnosis.

**Our fraud example:** `resolution_code` is entered by a human analyst *after* fraud is confirmed. In production, when a transaction arrives in real time, this field is blank. The model will never see a meaningful value for it. But if we include it during training, the model figures out: "code 3 = fraud" — and achieves 99%+ accuracy that will vanish entirely in production.


In [ ]:
# ============================================================
# CELL 3: Demonstrate Target Leakage — the 99% accuracy trap
# ============================================================

# ---- Encode categorical column for modelling ----
df_encoded = df.copy()
df_encoded['merchant_cat_code'] = df_encoded['merchant_category'].astype('category').cat.codes

# ---- Feature sets ----
LEAKY_FEATURES = [
    'amount', 'hour_of_day', 'distance_from_home_km',
    'merchant_cat_code', 'resolution_code'   # <-- resolution_code is the leaky one
]

CLEAN_FEATURES = [
    'amount', 'hour_of_day', 'distance_from_home_km',
    'merchant_cat_code'                      # <-- resolution_code removed
]

TARGET = 'is_fraud'

# ---- Split data 80/20 — note: simple random split for now ----
# (We will show why even this split is flawed in the Group Leakage section)
X_leaky = df_encoded[LEAKY_FEATURES]
X_clean = df_encoded[CLEAN_FEATURES]
y = df_encoded[TARGET]

X_leaky_train, X_leaky_test, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_clean_train, X_clean_test, _, _ = train_test_split(
    X_clean, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# ---- Train Logistic Regression on LEAKY features ----
model_leaky = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_leaky.fit(X_leaky_train, y_train)
auc_leaky = roc_auc_score(y_test, model_leaky.predict_proba(X_leaky_test)[:, 1])

# ---- Train Logistic Regression on CLEAN features ----
model_clean = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_clean.fit(X_clean_train, y_train)
auc_clean = roc_auc_score(y_test, model_clean.predict_proba(X_clean_test)[:, 1])

print('=' * 55)
print('TARGET LEAKAGE DEMONSTRATION')
print('=' * 55)
print(f'Model WITH leaky feature  (resolution_code): AUC = {auc_leaky:.4f}')
print(f'Model WITHOUT leaky feature (clean):         AUC = {auc_clean:.4f}')
print()
print('The leaky model looks INCREDIBLE in testing.')
print('In production it will score ~0.5 AUC because')
print('resolution_code does not exist at prediction time.')

In [ ]:
# ============================================================
# CELL 4: Feature importance — spotting the leaky feature
# ============================================================
# Random Forest gives us feature importances — a leaky feature will
# almost always dominate importance scores, which is a red flag.

rf_leaky = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_leaky.fit(X_leaky_train, y_train)

# Build a clean importance DataFrame for plotting
importance_df = pd.DataFrame({
    'feature':    LEAKY_FEATURES,
    'importance': rf_leaky.feature_importances_
}).sort_values('importance', ascending=True)  # ascending=True for horizontal bar chart

# ---- Plot ----
fig, ax = plt.subplots(figsize=(8, 4))

# Colour the leaky feature red so it stands out immediately
colors = ['#d62728' if feat == 'resolution_code' else '#1f77b4'
          for feat in importance_df['feature']]

ax.barh(importance_df['feature'], importance_df['importance'], color=colors)
ax.set_xlabel('Feature Importance (Random Forest)', fontsize=11)
ax.set_title('Feature Importance — Red = Leaky Feature\n'
             'A single feature dominating importance is a major red flag',
             fontsize=12)

# Add a legend manually
red_patch = mpatches.Patch(color='#d62728', label='Leaky feature (resolution_code)')
blue_patch = mpatches.Patch(color='#1f77b4', label='Legitimate feature')
ax.legend(handles=[red_patch, blue_patch], loc='lower right')

plt.tight_layout()
plt.show()

print('\nKey Diagnostic Rule:')
print('If one feature has >>50% importance, ask yourself:')
print('"Is this feature available at prediction time?"')

---
## Type 2: Train-Test Contamination

**Plain English definition:** You accidentally let information from the test set influence how you prepare (preprocess) your training data. The most common form is fitting a scaler or imputer on the *full* dataset before splitting into train and test.

**Why it's subtle:** The contamination doesn't add a leaky *feature*. It leaks *statistics* — the mean, standard deviation, or median of the test set flows back into the training process. The model is subtly tuned to the test set without you knowing.

**Concrete example:**
- You have 10,000 transactions. Test set contains 2,000.
- You compute `StandardScaler` on all 10,000 rows.
- The scaler's `mean_` and `scale_` are influenced by 2,000 test rows.
- When you evaluate on the test set, its own statistics were baked into the preprocessing.
- In production, the scaler was computed with test data it has never seen → mismatch.


In [ ]:
# ============================================================
# CELL 5: Demonstrate Train-Test Contamination
# ============================================================

X = df_encoded[CLEAN_FEATURES].copy()
y = df_encoded[TARGET].copy()

# ---- SPLIT first (this is the correct order) ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# ==============================================================
# BAD APPROACH: fit scaler on FULL dataset (before split)
# This is what many beginners do — it looks harmless but isn't
# ==============================================================
scaler_contaminated = StandardScaler()
# .fit() computes mean and std using ALL rows — including test rows!
scaler_contaminated.fit(X)  # <-- BUG: using X (full), not X_train

X_train_cont = scaler_contaminated.transform(X_train)
X_test_cont  = scaler_contaminated.transform(X_test)

model_cont = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_cont.fit(X_train_cont, y_train)
auc_contaminated = roc_auc_score(y_test, model_cont.predict_proba(X_test_cont)[:, 1])

# ==============================================================
# GOOD APPROACH: fit scaler on TRAINING data only
# ==============================================================
scaler_correct = StandardScaler()
# .fit_transform() computes stats and transforms — training data only
X_train_corr = scaler_correct.fit_transform(X_train)  # <-- CORRECT
# .transform() uses stats from training data to scale test data
X_test_corr  = scaler_correct.transform(X_test)       # <-- CORRECT

model_corr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_corr.fit(X_train_corr, y_train)
auc_correct = roc_auc_score(y_test, model_corr.predict_proba(X_test_corr)[:, 1])

print('=' * 55)
print('TRAIN-TEST CONTAMINATION DEMONSTRATION')
print('=' * 55)
print(f'Scaler fit on FULL data (contaminated): AUC = {auc_contaminated:.4f}')
print(f'Scaler fit on TRAIN data only (correct): AUC = {auc_correct:.4f}')
print()
print('On small datasets the gap is modest.')
print('On production data with distribution shift, the gap can be severe.')
print()
# Show that the scaler statistics ARE different
print('Contaminated scaler mean (amount):', round(scaler_contaminated.mean_[0], 4))
print('Correct scaler mean (amount):     ', round(scaler_correct.mean_[0], 4))
print('(Different because contaminated version includes test set rows)')

In [ ]:
# ============================================================
# CELL 6: Visualise the contamination gap — scaler mean comparison
# ============================================================

feature_names = CLEAN_FEATURES

# Collect scaler means for each feature under both approaches
x_pos = np.arange(len(feature_names))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Left panel: Scaler mean comparison ----
ax = axes[0]
bars1 = ax.bar(x_pos - width/2, scaler_contaminated.mean_, width,
               label='Contaminated (fit on full data)', color='#d62728', alpha=0.8)
bars2 = ax.bar(x_pos + width/2, scaler_correct.mean_, width,
               label='Correct (fit on train only)', color='#2ca02c', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Scaler Mean Value')
ax.set_title('Scaler Means: Contaminated vs Correct\n'
             'Slight differences propagate to shifted model weights', fontsize=10)
ax.legend(fontsize=9)

# ---- Right panel: AUC comparison bar ----
ax2 = axes[1]
models = ['Contaminated\n(scaler on full data)', 'Correct\n(scaler on train only)']
aucs   = [auc_contaminated, auc_correct]
bar_colors = ['#d62728', '#2ca02c']
bars = ax2.bar(models, aucs, color=bar_colors, alpha=0.85, width=0.5)
ax2.set_ylim(0.5, 1.0)
ax2.set_ylabel('ROC-AUC on Test Set')
ax2.set_title('AUC Comparison\nContamination can inflate test performance', fontsize=10)
# Annotate bar heights
for bar, auc in zip(bars, aucs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{auc:.4f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Type 2: Train-Test Contamination', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Type 3: Temporal Leakage

**Plain English definition:** You use information from the *future* to make predictions about the *past*. This happens when features are computed from the full time range of data, so future events influence features that supposedly describe earlier events.

**Stock market analogy:** Imagine you want to predict whether a stock will rise tomorrow. You compute a feature: "was this stock in the top 10% performers for the full year?" — but you compute that rank using the *entire year's* data. To make that prediction for January, you used December's returns. That's temporal leakage.

**Our fraud example:** A common feature is `rolling_fraud_rate` — the percentage of recent transactions from this merchant category that were fraudulent. If you compute this using the *full* dataset, then a transaction in January will have a fraud-rate feature that includes fraud events from March, June, and December. Information from the future is predicting the past.

**Prevention:** Always compute rolling/aggregate features using only data that existed *before* the prediction timestamp. Use `.expanding()` or `.shift()` in pandas. For cross-validation, use `TimeSeriesSplit`.


In [ ]:
# ============================================================
# CELL 7: Temporal Leakage — future fraud rate contaminates features
# ============================================================

# Sort by timestamp — essential before computing rolling features
df_time = df.sort_values('timestamp').reset_index(drop=True)

# ---- BAD: Compute rolling fraud rate using FULL dataset group mean ----
# This computes the overall fraud rate for each merchant category
# using ALL transactions — including ones that happen in the future.
category_fraud_rate_full = df_time.groupby('merchant_category')['is_fraud'].transform('mean')
df_time['leaky_merchant_fraud_rate'] = category_fraud_rate_full

# ---- GOOD: Compute rolling fraud rate using ONLY past data ----
# .expanding() computes cumulative stats up to (but not including) current row
# .shift(1) ensures we use data up to the previous row (true past only)
df_time['clean_merchant_fraud_rate'] = (
    df_time.groupby('merchant_category')['is_fraud']
    .transform(lambda x: x.shift(1).expanding().mean())  # shift(1) = exclude current row
)
# Fill NaN (first occurrence of each category has no history) with overall prior
prior_rate = df_time['is_fraud'].mean()
df_time['clean_merchant_fraud_rate'] = df_time['clean_merchant_fraud_rate'].fillna(prior_rate)

# ---- Encode merchant category for modelling ----
df_time['merchant_cat_code'] = df_time['merchant_category'].astype('category').cat.codes

# ---- Time-based train/test split: first 80% of time = train, last 20% = test ----
# This mirrors real deployment: you train on historical data and predict on future data.
split_idx = int(len(df_time) * 0.8)
split_date = df_time.iloc[split_idx]['timestamp']
print(f'Temporal split date: {split_date.date()}')
print(f'Train rows: {split_idx:,} | Test rows: {len(df_time) - split_idx:,}')

base_features = ['amount', 'hour_of_day', 'distance_from_home_km', 'merchant_cat_code']

# ---- Model 1: Using LEAKY rolling feature ----
leaky_feats = base_features + ['leaky_merchant_fraud_rate']
X_leaky_tmp = df_time[leaky_feats]
y_time = df_time['is_fraud']
X_tr_l, X_te_l = X_leaky_tmp.iloc[:split_idx], X_leaky_tmp.iloc[split_idx:]
y_tr, y_te = y_time.iloc[:split_idx], y_time.iloc[split_idx:]

m_leaky = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
m_leaky.fit(X_tr_l, y_tr)
auc_temporal_leaky = roc_auc_score(y_te, m_leaky.predict_proba(X_te_l)[:, 1])

# ---- Model 2: Using CLEAN rolling feature ----
clean_feats = base_features + ['clean_merchant_fraud_rate']
X_clean_tmp = df_time[clean_feats]
X_tr_c, X_te_c = X_clean_tmp.iloc[:split_idx], X_clean_tmp.iloc[split_idx:]

m_clean = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
m_clean.fit(X_tr_c, y_tr)
auc_temporal_clean = roc_auc_score(y_te, m_clean.predict_proba(X_te_c)[:, 1])

print(f'\nTemporal Leaky model AUC:  {auc_temporal_leaky:.4f}')
print(f'Temporal Clean model AUC:  {auc_temporal_clean:.4f}')

In [ ]:
# ============================================================
# CELL 8: Visualise leaky vs clean rolling fraud rate over time
# ============================================================

# Sample one merchant category for clarity
category_sample = 'online'
cat_df = df_time[df_time['merchant_category'] == category_sample].copy()
cat_df = cat_df.iloc[:200]  # first 200 transactions in this category

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# ---- Top panel: leaky vs clean rolling fraud rate ----
ax = axes[0]
ax.plot(cat_df.index, cat_df['leaky_merchant_fraud_rate'],
        color='#d62728', linewidth=1.5, label='Leaky rate (uses future data)')
ax.plot(cat_df.index, cat_df['clean_merchant_fraud_rate'],
        color='#2ca02c', linewidth=1.5, label='Clean rate (past only)')
ax.set_ylabel('Rolling Fraud Rate')
ax.set_title(f'Rolling Fraud Rate — "online" Category (first 200 transactions)\n'
             f'Leaky version is a flat line (uses all data equally), '
             f'clean version evolves over time', fontsize=10)
ax.legend()
ax.axhline(y=prior_rate, color='grey', linestyle='--', linewidth=0.8, label='Prior rate')

# ---- Bottom panel: actual fraud labels ----
ax2 = axes[1]
ax2.scatter(cat_df.index, cat_df['is_fraud'],
            c=cat_df['is_fraud'].map({0: '#2ca02c', 1: '#d62728'}),
            alpha=0.5, s=15)
ax2.set_ylabel('Is Fraud (0/1)')
ax2.set_xlabel('Transaction Index (chronological)')
ax2.set_title('Actual Fraud Labels', fontsize=10)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Legitimate', 'Fraud'])

plt.suptitle('Type 3: Temporal Leakage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Type 4: Group Leakage

**Plain English definition:** The same real-world *entity* (a user, a patient, a household) has records in both the training set and the test set. The model memorises this entity's behaviour patterns during training and then "recognises" the same entity in the test set — giving falsely high performance.

**Medical analogy:** A patient has 10 doctor visits over two years. You randomly split 80/20 — 8 visits in training, 2 in test. The model learns that *this specific patient's pattern* is associated with their diagnosis. On the test set, it recognises the same patient and "cheats." In a new hospital, this patient has never been seen and the model fails.

**Our fraud example:** User 12345 makes 50 transactions. A random 80/20 split puts 40 in training and 10 in test. The model learns User 12345's spending patterns in training and then easily "identifies" those same patterns in the 10 test transactions. But in production, *new* users will appear — users the model has never seen.

**Math:** With 500 users and 50 transactions per user, a random 80/20 split will put ~40 transactions from EVERY user in training and ~10 in test. Almost 100% of test users were already seen in training. This is pure group leakage.

**Prevention:** Split at the *entity level*. Put 80% of users entirely in training and 20% entirely in test.


In [ ]:
# ============================================================
# CELL 9: Group Leakage — row split vs user-level split
# ============================================================

# Use encoded DataFrame with no leaky features
df_model = df_encoded.copy()
features = CLEAN_FEATURES
y_all = df_model['is_fraud']
X_all = df_model[features]

# ==============================================================
# LEAKY SPLIT: random row-level 80/20
# Almost every user appears in both train AND test
# ==============================================================
X_tr_row, X_te_row, y_tr_row, y_te_row, idx_tr_row, idx_te_row = train_test_split(
    X_all, y_all, df_model.index,
    test_size=0.2, random_state=RANDOM_STATE, stratify=y_all
)

# Count how many test users also appear in training
train_users_row = set(df_model.loc[idx_tr_row, 'user_id'])
test_users_row  = set(df_model.loc[idx_te_row, 'user_id'])
overlap_row = len(train_users_row & test_users_row)

model_row = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_row.fit(X_tr_row, y_tr_row)
auc_row = roc_auc_score(y_te_row, model_row.predict_proba(X_te_row)[:, 1])

# ==============================================================
# CORRECT SPLIT: split at user level
# 20% of users are entirely held out for testing
# ==============================================================
all_users = df_model['user_id'].unique()
np.random.seed(RANDOM_STATE)
# Shuffle users and take 80% for training, 20% for testing
np.random.shuffle(all_users)
split_user_idx = int(len(all_users) * 0.8)
train_users_set = set(all_users[:split_user_idx])
test_users_set  = set(all_users[split_user_idx:])

train_mask = df_model['user_id'].isin(train_users_set)
test_mask  = df_model['user_id'].isin(test_users_set)

X_tr_usr = X_all[train_mask];  y_tr_usr = y_all[train_mask]
X_te_usr = X_all[test_mask];   y_te_usr = y_all[test_mask]

# Zero overlap by construction
overlap_usr = len(train_users_set & test_users_set)

model_usr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_usr.fit(X_tr_usr, y_tr_usr)
auc_usr = roc_auc_score(y_te_usr, model_usr.predict_proba(X_te_usr)[:, 1])

print('=' * 55)
print('GROUP LEAKAGE DEMONSTRATION')
print('=' * 55)
print(f'Row-level split  — user overlap: {overlap_row}/{len(test_users_row)} test users '
      f'({overlap_row/len(test_users_row):.0%}) | AUC = {auc_row:.4f}')
print(f'User-level split — user overlap: {overlap_usr}/{len(test_users_set)} test users '
      f'({overlap_usr/len(test_users_set):.0%}) | AUC = {auc_usr:.4f}')
print()
print('Row-level split inflates AUC because the model memorises user patterns.')
print('User-level split tests genuine generalisation to unseen users.')

---
## Systematic Prevention: The Leakage Audit Function

Before training any model, run a structured audit of your dataset. The function below checks for several common leakage signals automatically.


In [ ]:
# ============================================================
# CELL 10: Leakage detection checklist function
# ============================================================

def audit_for_leakage(df: pd.DataFrame, target_col: str,
                      timestamp_col: str = None,
                      entity_col: str = None,
                      suspicious_keywords: list = None) -> dict:
    """
    Runs a leakage audit on a DataFrame.

    Parameters
    ----------
    df               : The full dataset before any train/test split
    target_col       : Name of the target / label column
    timestamp_col    : Name of the timestamp column (for temporal checks)
    entity_col       : Name of the entity/group column (for group leakage)
    suspicious_keywords : List of keywords that suggest post-event features

    Returns
    -------
    Dictionary of findings
    """

    if suspicious_keywords is None:
        # These keywords often indicate a feature was created AFTER the event
        suspicious_keywords = [
            'resolution', 'outcome', 'final', 'result', 'confirmed',
            'approved', 'rejected', 'closed', 'settled', 'days_to',
            'post_', 'after_', 'review', 'flag'
        ]

    findings = {}
    feature_cols = [c for c in df.columns if c != target_col]

    # ---- Check 1: Perfect or near-perfect correlation with target ----
    # Any feature with |correlation| > 0.9 is extremely suspicious
    numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns
    high_corr = {}
    for col in numeric_cols:
        if col in [timestamp_col, entity_col]:
            continue
        corr = abs(df[col].corr(df[target_col]))
        if corr > 0.7:  # threshold: adjust based on domain knowledge
            high_corr[col] = round(corr, 4)
    findings['high_correlation_with_target'] = high_corr

    # ---- Check 2: Column names containing suspicious keywords ----
    suspicious_cols = [
        col for col in feature_cols
        if any(kw in col.lower() for kw in suspicious_keywords)
    ]
    findings['suspicious_column_names'] = suspicious_cols

    # ---- Check 3: Features with zero variance in one class (perfect separator) ----
    # A legitimate feature should have some overlap between fraud and not-fraud
    perfect_separators = []
    for col in numeric_cols:
        class0_vals = df[df[target_col] == 0][col]
        class1_vals = df[df[target_col] == 1][col]
        if class0_vals.max() < class1_vals.min() or class1_vals.max() < class0_vals.min():
            perfect_separators.append(col)
    findings['perfect_class_separators'] = perfect_separators

    # ---- Check 4: Group overlap check ----
    if entity_col:
        avg_transactions_per_entity = len(df) / df[entity_col].nunique()
        findings['avg_transactions_per_entity'] = round(avg_transactions_per_entity, 1)
        findings['group_leakage_risk'] = (
            'HIGH — random split will cause overlap' if avg_transactions_per_entity > 5
            else 'LOW — entities rarely repeat'
        )

    # ---- Check 5: Temporal column detected ----
    findings['temporal_column_found'] = timestamp_col is not None
    findings['recommendation_temporal'] = (
        'Use date-based train/test split and TimeSeriesSplit for CV'
        if timestamp_col else 'No timestamp column found — verify this is not needed'
    )

    return findings


# ---- Run the audit ----
audit_results = audit_for_leakage(
    df=df_encoded,
    target_col='is_fraud',
    timestamp_col='timestamp',
    entity_col='user_id'
)

print('=' * 60)
print('DATA LEAKAGE AUDIT REPORT')
print('=' * 60)
for check, result in audit_results.items():
    print(f'\n[{check.upper()}]')
    print(f'  {result}')

---
## The Prevention Checklist

Before training any model, go through this checklist:

| # | Question | What to do if "Yes" |
|---|----------|---------------------|
| 1 | Does any feature have correlation > 0.9 with the target? | Investigate — is it derived from the target? |
| 2 | Does any feature name contain post-event keywords (resolution, outcome, confirmed)? | Verify temporal relationship — was this created before or after the event? |
| 3 | Are you fitting any transformer (scaler, imputer, encoder) before splitting data? | Always split FIRST, then fit transformers on training data only |
| 4 | Do you have timestamp data? | Use date-based split and TimeSeriesSplit for cross-validation |
| 5 | Does the same entity appear many times? | Split by entity, not by row |
| 6 | Does your model score suspiciously higher (>10%) on test vs what makes domain sense? | Investigate all of the above |
| 7 | Are you doing feature selection, dimensionality reduction, or anomaly detection before splitting? | These must all happen inside a Pipeline, on training data only |


In [ ]:
# ============================================================
# CELL 11: Correct temporal split using TimeSeriesSplit CV
# ============================================================

# TimeSeriesSplit ensures each validation fold only sees future data
# relative to what the model was trained on — no temporal leakage possible
# in cross-validation.

df_time_model = df_time.copy()
df_time_model['merchant_cat_code'] = (
    df_time_model['merchant_category'].astype('category').cat.codes
)

X_ts = df_time_model[CLEAN_FEATURES].values
y_ts = df_time_model['is_fraud'].values

# n_splits=5 means 5 expanding-window folds
# Fold 1: train on rows 0-1666, test on rows 1666-3333
# Fold 2: train on rows 0-3333, test on rows 3333-5000
# ... and so on (training window always expands, never shrinks)
tscv = TimeSeriesSplit(n_splits=5)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

fold_aucs = []
fold_info = []

for fold_num, (train_idx, test_idx) in enumerate(tscv.split(X_ts), start=1):
    X_fold_train, X_fold_test = X_ts[train_idx], X_ts[test_idx]
    y_fold_train, y_fold_test = y_ts[train_idx], y_ts[test_idx]

    pipe.fit(X_fold_train, y_fold_train)  # Pipeline fits scaler on training fold only
    proba = pipe.predict_proba(X_fold_test)[:, 1]
    auc   = roc_auc_score(y_fold_test, proba)
    fold_aucs.append(auc)
    fold_info.append({'fold': fold_num,
                      'train_size': len(train_idx),
                      'test_size':  len(test_idx),
                      'auc':        round(auc, 4)})

fold_df = pd.DataFrame(fold_info)
print('TimeSeriesSplit Cross-Validation Results')
print('=' * 45)
print(fold_df.to_string(index=False))
print(f'\nMean AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')
print()
print('Note: Training windows expand in size (no look-ahead bias).')
print('The model NEVER sees future transactions during training.')

In [ ]:
# ============================================================
# CELL 12: Grand summary visualisation — all four leakage types
# ============================================================

# Collect all AUC scores across our experiments
leakage_scenarios = {
    'Target Leakage\n(leaky)':          auc_leaky,
    'Target Leakage\n(clean)':          auc_clean,
    'Train-Test Contam.\n(leaky)':      auc_contaminated,
    'Train-Test Contam.\n(clean)':      auc_correct,
    'Temporal Leakage\n(leaky)':        auc_temporal_leaky,
    'Temporal Leakage\n(clean)':        auc_temporal_clean,
    'Group Leakage\n(leaky)':           auc_row,
    'Group Leakage\n(clean)':           auc_usr,
}

labels = list(leakage_scenarios.keys())
values = list(leakage_scenarios.values())

# Colour leaky bars red, clean bars green
bar_colors = ['#d62728' if 'leaky' in lbl else '#2ca02c' for lbl in labels]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(labels)), values, color=bar_colors, alpha=0.85, edgecolor='white')

# Add a horizontal line for "realistic" performance expectation
ax.axhline(y=0.80, color='navy', linestyle='--', linewidth=1.2,
           label='Realistic AUC ceiling (~0.80)')

# Annotate each bar with its AUC value
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.004,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0.5, 1.02)
ax.set_ylabel('ROC-AUC Score', fontsize=11)
ax.set_title('Leaky vs Clean Models — AUC Comparison Across All Four Leakage Types\n'
             'Red = leaky model | Green = correct model | Dashed = realistic performance ceiling',
             fontsize=12)

red_patch   = mpatches.Patch(color='#d62728', label='Leaky model')
green_patch = mpatches.Patch(color='#2ca02c', label='Clean model')
ax.legend(handles=[red_patch, green_patch], loc='upper right')

plt.tight_layout()
plt.show()

print('\nKey takeaway:')
print('Leaky models look extraordinary in testing but deliver ordinary or worse in production.')
print('The gap between red and green is the gap between "looks great" and "actually works".')

---
## The Final Prevention Framework

```
STEP 1  — Understand your data timeline
         Draw a timeline: when does each feature get its value?
         If the value is set AFTER the event → suspect leakage.

STEP 2  — Split BEFORE fitting anything
         train_test_split() is your FIRST operation after loading data.
         Never fit a scaler, imputer, or encoder on the full dataset.

STEP 3  — Use sklearn Pipelines (see Notebook 10)
         Pipelines enforce the correct fit-on-train / transform-on-test
         order automatically — even inside cross-validation.

STEP 4  — Use the right CV strategy
         Time-series data?  → TimeSeriesSplit
         Grouped data?      → GroupKFold
         Regular data?      → StratifiedKFold

STEP 5  — Audit feature importance
         Any single feature with >50% importance is a red flag.
         Ask: "Could this feature exist without knowing the target?"

STEP 6  — Validate against a hold-out set you NEVER touched
         Separate from your test set used in cross-validation.
         This is your "production simulation".
```


In [ ]:
# ============================================================
# CELL 13: GroupKFold for grouped data — correct CV strategy
# ============================================================

from sklearn.model_selection import GroupKFold

X_grp = df_encoded[CLEAN_FEATURES].values
y_grp = df_encoded['is_fraud'].values
groups = df_encoded['user_id'].values  # group labels = user IDs

# GroupKFold ensures no user appears in both train and validation
# within any single fold — the correct approach for user-level leakage
gkf = GroupKFold(n_splits=5)

group_fold_aucs = []
group_fold_info = []

for fold_num, (train_idx, test_idx) in enumerate(gkf.split(X_grp, y_grp, groups), start=1):
    X_tr_g, X_te_g = X_grp[train_idx], X_grp[test_idx]
    y_tr_g, y_te_g = y_grp[train_idx], y_grp[test_idx]

    # Overlap check: zero unique users should be shared
    train_u = set(groups[train_idx])
    test_u  = set(groups[test_idx])
    overlap  = len(train_u & test_u)

    pipe_grp = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ])
    pipe_grp.fit(X_tr_g, y_tr_g)
    auc_g = roc_auc_score(y_te_g, pipe_grp.predict_proba(X_te_g)[:, 1])
    group_fold_aucs.append(auc_g)
    group_fold_info.append({
        'fold': fold_num,
        'train_users': len(train_u),
        'test_users':  len(test_u),
        'user_overlap': overlap,
        'auc': round(auc_g, 4)
    })

print('GroupKFold Cross-Validation Results')
print('=' * 55)
print(pd.DataFrame(group_fold_info).to_string(index=False))
print(f'\nMean AUC: {np.mean(group_fold_aucs):.4f} ± {np.std(group_fold_aucs):.4f}')
print()
print('User overlap = 0 in EVERY fold — no group leakage possible.')

---
## Self-Check Questions

Test your understanding before moving on. Try to answer each question in your own words *before* expanding the answer.

---

**Question 1:** A feature `days_to_resolution` (the number of days between the transaction date and the date it was formally resolved as fraud or not) is in your dataset. Is this leaky? Justify your answer.

<details>
<summary>Show Answer</summary>

**Yes, this is leaky — Target Leakage.** The feature `days_to_resolution` only has a meaningful value *after* the transaction has been reviewed and resolved. At prediction time (when a transaction just occurred), this value is unknown — it would be 0 or null for all new transactions. Furthermore, the value of this feature is inherently tied to whether fraud occurred: fraudulent transactions may have longer or shorter resolution times than legitimate ones. Including it allows the model to learn the answer (fraud/not fraud) from a feature that doesn't exist at prediction time. Remove it.

</details>

---

**Question 2:** You get 98% AUC on your test set but 71% AUC in production. What is the most likely cause?

<details>
<summary>Show Answer</summary>

The most likely cause is **data leakage** — specifically one or more of:

1. **Target Leakage:** A post-event feature was included in training (e.g., `resolution_code`, `fraud_confirmed_flag`). At prediction time this feature is empty/null, so the model collapses to near-random.
2. **Train-Test Contamination:** A preprocessing step (scaler, imputer) was fit on the full dataset — the model was subtly tuned to test-set statistics that don't generalise.
3. **Temporal Leakage:** Rolling/aggregate features were computed using the full time range — the "future" is not available in production.

The severity of the drop (98% → 71%) most strongly suggests Target Leakage — a feature that *is* the answer in testing becomes missing/constant in production, and the model loses almost all predictive power.

</details>

---

**Question 3:** You have 50,000 transactions from 1,000 users (50 transactions per user). You randomly split 80/20. Approximately how many users appear in BOTH train and test? Why is this a problem?

<details>
<summary>Show Answer</summary>

**Almost all 1,000 users appear in both.** With 50 transactions per user, an 80/20 random split will assign roughly 40 transactions from each user to training and 10 to testing. This means ~100% of users are in both sets.

This is a problem because:
- The model learns each specific user's spending patterns (e.g., "User 12345 always spends exactly $42 at grocery stores on Tuesdays").
- When User 12345 appears in the test set, the model "recognises" this user and makes accurate predictions based on memorised patterns.
- In production, genuinely new users will appear — users the model has never seen. The memorised patterns don't help, and performance drops significantly.

**Correct approach:** Split at the user level — put 800 users entirely in training and 200 users entirely in testing.

</details>

---

**Question 4:** How does using a sklearn Pipeline prevent Train-Test Contamination?

<details>
<summary>Show Answer</summary>

A sklearn Pipeline enforces the following behaviour:

- **`pipeline.fit(X_train, y_train)`** calls `.fit_transform()` on every preprocessing step using *only* `X_train`. The scaler computes its mean and std from training data only. The imputer computes its median from training data only. No test data is involved.
- **`pipeline.predict(X_test)`** calls `.transform()` on each step using the statistics already computed from training data. The test data is transformed but never *fits* anything.

This is the key guarantee: fit parameters (mean, std, median, encoder categories) are always computed on training data and applied to test data — not the other way around. The Pipeline makes this impossible to accidentally break, even inside cross-validation loops.

</details>


In [ ]:
# ============================================================
# CELL 14: Summary table — all AUC results side by side
# ============================================================

summary_data = {
    'Leakage Type':    ['Target', 'Target',
                        'Train-Test Contam.', 'Train-Test Contam.',
                        'Temporal', 'Temporal',
                        'Group', 'Group'],
    'Version':         ['Leaky', 'Clean'] * 4,
    'AUC':             [auc_leaky, auc_clean,
                        auc_contaminated, auc_correct,
                        auc_temporal_leaky, auc_temporal_clean,
                        auc_row, auc_usr]
}

summary_df = pd.DataFrame(summary_data)
summary_df['AUC'] = summary_df['AUC'].round(4)

# Pivot for readability
pivot = summary_df.pivot(index='Leakage Type', columns='Version', values='AUC')
pivot['Performance Inflation'] = (pivot['Leaky'] - pivot['Clean']).round(4)

print('Data Leakage — AUC Summary Table')
print('=' * 50)
print(pivot.to_string())
print()
print('"Performance Inflation" = how much the leaky model')
print('artificially overstates true performance.')
print('This inflation disappears in production.')